### Advanced Problems with Solutions — The `__prepare__` Method

This notebook continues the topic of metaclass `__prepare__`, but it is deliberately written in a **tutorial style**.

Instead of jumping directly from a problem statement to a large solution, we will proceed in small steps:

1. observe a behavior,
2. ask what is happening,
3. build a small piece,
4. inspect the result,
5. add one more requirement,
6. finish with a complete solution.

The examples are independent of the earlier exercise notebook and focus on different advanced uses of the class namespace.

We will use the same core idea throughout the notebook:

When Python starts creating a class, it needs a namespace in which to execute the class body.

That namespace is obtained from the metaclass `__prepare__` method.

The important point is that the object returned by `__prepare__` is not just a container that we inspect later.

It is the namespace Python actively uses **while the class body is executing**.

This means a custom prepared namespace can do things that are impossible to reconstruct reliably after the class body has finished.

For example, it can:

- observe assignments as they happen,
- reject a name before it overwrites another value,
- expose temporary helper names,
- intercept annotation writes,
- enforce conventions,
- attach metadata to functions before the class exists.

We will explore each of these ideas separately.

In [1]:
def check(condition, message="check failed"):
    if not condition:
        raise AssertionError(message)

def check_equal(actual, expected, message=None):
    if actual != expected:
        raise AssertionError(
            message or f"expected {expected!r}, got {actual!r}"
        )

print("Helpers ready.")

Helpers ready.


## Problem 1 — Watch the class namespace being populated

Let us start by making the namespace itself visible.

We want to see every assignment Python performs while it executes a class body.

A regular dictionary already supports everything Python needs for ordinary class-body assignments.

So the smallest useful experiment is to subclass `dict` and override `__setitem__`.

In [2]:
class WatchingDict(dict):
    def __setitem__(self, key, value):
        print(f"WRITE: {key!r} -> {value!r}")
        super().__setitem__(key, value)

Now we need Python to use this dictionary for a class body.

That is exactly what `__prepare__` controls.

In [3]:
class WatchingMeta(type):
    @classmethod
    def __prepare__(mcls, name, bases, **kwargs):
        print(f"PREPARE: {name}")
        return WatchingDict()

Let us define a small class.

Do not focus only on the names we explicitly write.

Also look for names Python inserts automatically.

In [4]:
class WatchedExample(metaclass=WatchingMeta):
    x = 10
    y = 20

    def total(self):
        return self.x + self.y

PREPARE: WatchedExample
WRITE: '__module__' -> '__main__'
WRITE: '__qualname__' -> 'WatchedExample'
WRITE: '__firstlineno__' -> 1
WRITE: 'x' -> 10
WRITE: 'y' -> 20
WRITE: 'total' -> <function WatchedExample.total at 0x000001DD7EA44EA0>
WRITE: '__static_attributes__' -> ()


We can see that the namespace receives entries such as `__module__` and `__qualname__` in addition to our own attributes.

A method definition is also ultimately an assignment: the function object is created and stored under the method name.

This gives us our first advanced design principle:

> If a rule concerns *assignments made by the class body*, a custom prepared namespace is often the earliest and most precise place to enforce it.

## Problem 2 — Validate class options before the body runs

Class header keyword arguments are passed to `__prepare__`.

That means `__prepare__` can reject invalid configuration **before Python executes the body at all**.

We will prove that with a side effect.

In [5]:
prepare_side_effects = []

class ModeMeta(type):
    @classmethod
    def __prepare__(mcls, name, bases, **kwargs):
        mode = kwargs.get("mode", "strict")

        if mode not in {"strict", "relaxed"}:
            raise ValueError(f"unsupported mode: {mode!r}")

        return {"configured_mode": mode}

    def __new__(mcls, name, bases, namespace, **kwargs):
        kwargs.pop("mode", None)
        return super().__new__(mcls, name, bases, namespace)

The next class is valid, so its body should execute normally.

In [6]:
class StrictService(metaclass=ModeMeta, mode="strict"):
    prepare_side_effects.append("StrictService body executed")
    value = 100

check_equal(StrictService.configured_mode, "strict")
print(prepare_side_effects)

['StrictService body executed']


Now we will deliberately use an invalid mode.

If `__prepare__` runs before the class body, the body should never append its marker.

In [7]:
bad_mode_source = '''
class InvalidService(metaclass=ModeMeta, mode="impossible"):
    prepare_side_effects.append("THIS MUST NOT RUN")
    value = 200
'''

try:
    exec(bad_mode_source, globals(), globals())
except ValueError as exc:
    print("Caught:", exc)
else:
    raise AssertionError("expected ValueError")

check("THIS MUST NOT RUN" not in prepare_side_effects)
print(prepare_side_effects)

Caught: unsupported mode: 'impossible'
['StrictService body executed']


Notice the timing.

The failure happened before the body could perform its side effect.

This is useful when a class option changes the rules under which the body is allowed to execute.

## Problem 3 — Enforce `snake_case` names as they are declared

Suppose we are writing a framework in which public class attributes must use `snake_case`.

We could inspect the final class dictionary later.

But if we enforce the rule in the prepared namespace, the error points directly to the class-definition step that violated the policy.

We first need a small name checker.

Double-underscore implementation names should not be rejected.

In [8]:
import re

_snake_case = re.compile(r"^[a-z][a-z0-9_]*$")

class SnakeCaseNamespace(dict):
    def __setitem__(self, key, value):
        is_dunder = key.startswith("__") and key.endswith("__")

        if not is_dunder and not _snake_case.fullmatch(key):
            raise NameError(
                f"public class name {key!r} must use snake_case"
            )

        super().__setitem__(key, value)

Now we connect it to a metaclass.

In [9]:
class SnakeCaseMeta(type):
    @classmethod
    def __prepare__(mcls, name, bases, **kwargs):
        return SnakeCaseNamespace()

A valid class should work normally.

In [10]:
class ValidSettings(metaclass=SnakeCaseMeta):
    request_timeout = 30
    max_retries = 5

check_equal(ValidSettings.request_timeout, 30)
print(vars(ValidSettings))

{'__module__': '__main__', '__firstlineno__': 1, 'request_timeout': 30, 'max_retries': 5, '__static_attributes__': (), '__dict__': <attribute '__dict__' of 'ValidSettings' objects>, '__weakref__': <attribute '__weakref__' of 'ValidSettings' objects>, '__doc__': None}


And an invalid public name should fail while the body is being executed.

In [11]:
invalid_name_source = '''
class InvalidSettings(metaclass=SnakeCaseMeta):
    requestTimeout = 30
'''

try:
    exec(invalid_name_source, globals(), globals())
except NameError as exc:
    print("Caught:", exc)
else:
    raise AssertionError("expected NameError")

Caught: public class name 'requestTimeout' must use snake_case


This type of namespace is best kept narrow.

It should enforce one clear invariant rather than becoming a general-purpose object that mysteriously changes normal Python behavior.

## Problem 4 — Instrument methods at assignment time

A function defined inside a class body is assigned to the namespace under its method name.

So a prepared namespace can notice functions at the moment they are declared.

Let us use that to wrap methods with a call counter.

We will only wrap plain Python functions.

We will leave special methods alone to avoid surprising behavior.

In [12]:
from functools import wraps
from types import FunctionType

def counted_function(func):
    calls = {"count": 0}

    @wraps(func)
    def wrapper(*args, **kwargs):
        calls["count"] += 1
        return func(*args, **kwargs)

    wrapper.__call_counter__ = calls
    return wrapper

class InstrumentingNamespace(dict):
    def __setitem__(self, key, value):
        is_special = key.startswith("__") and key.endswith("__")

        if isinstance(value, FunctionType) and not is_special:
            value = counted_function(value)

        super().__setitem__(key, value)

Now the metaclass only needs to return this namespace.

In [13]:
class InstrumentingMeta(type):
    @classmethod
    def __prepare__(mcls, name, bases, **kwargs):
        return InstrumentingNamespace()

Let us define and call a few methods.

In [14]:
class Calculator(metaclass=InstrumentingMeta):
    def add(self, a, b):
        return a + b

    def multiply(self, a, b):
        return a * b

calc = Calculator()

check_equal(calc.add(2, 3), 5)
check_equal(calc.add(10, 20), 30)
check_equal(calc.multiply(4, 5), 20)

print("add calls:", Calculator.add.__call_counter__["count"])
print("multiply calls:", Calculator.multiply.__call_counter__["count"])

check_equal(Calculator.add.__call_counter__["count"], 2)
check_equal(Calculator.multiply.__call_counter__["count"], 1)

add calls: 2
multiply calls: 1


The wrapping happened before the class object existed.

This is powerful, but it is also a reason to be conservative.

Automatically transforming every object stored in a class namespace can make code difficult to understand. A production framework should document such behavior very clearly.

## Problem 5 — Build a stateful class-body DSL

So far our prepared namespaces reacted to assignments.

But `__prepare__` can also place helper objects into the class namespace.

Those helpers can maintain state while the class body is executing.

We will build a tiny declaration system with sections.

The class body will look like this:

```python
class Configuration(...):
    section("network")
    host = "localhost"
    port = 8000

    section("security")
    tls = True
```

We want to remember which names were declared under each section.

The namespace itself will own the current section.

Its `section` method will switch that state.

In [15]:
class SectionNamespace(dict):
    def __init__(self):
        super().__init__()
        self.current_section = None
        self.sections = {}

    def choose_section(self, name):
        self.current_section = name
        self.sections.setdefault(name, [])

    def __setitem__(self, key, value):
        is_internal = key.startswith("__") and key.endswith("__")

        if (
            self.current_section is not None
            and not is_internal
            and key != "section"
        ):
            self.sections[self.current_section].append(key)

        super().__setitem__(key, value)

`__prepare__` creates one namespace object and inserts a bound helper that points back to that same object.

In [16]:
class SectionMeta(type):
    @classmethod
    def __prepare__(mcls, name, bases, **kwargs):
        ns = SectionNamespace()

        # Bypass our tracking hook for the framework helper itself.
        dict.__setitem__(ns, "section", ns.choose_section)
        return ns

    def __new__(mcls, name, bases, namespace, **kwargs):
        sections = {
            key: tuple(values)
            for key, values in namespace.sections.items()
        }

        normalized = dict(namespace)
        normalized.pop("section", None)

        cls = super().__new__(mcls, name, bases, normalized)
        cls.__sections__ = sections
        return cls

Now we can use the helper while the class body is running.

In [17]:
class AppConfiguration(metaclass=SectionMeta):
    section("network")
    host = "localhost"
    port = 8000

    section("security")
    tls = True
    token_ttl = 3600

check_equal(
    AppConfiguration.__sections__,
    {
        "network": ("host", "port"),
        "security": ("tls", "token_ttl"),
    },
)

check("section" not in AppConfiguration.__dict__)

print(AppConfiguration.__sections__)

{'network': ('host', 'port'), 'security': ('tls', 'token_ttl')}


This is an important pattern.

The helper exists for the *syntax of the class body*, but it does not need to survive as a class attribute.

`__prepare__` can therefore support a temporary DSL without polluting the final public API.

## Problem 6 — Permit an overwrite only when it is explicitly declared

A normal dictionary silently overwrites repeated keys.

A strict custom namespace could reject every duplicate.

But sometimes we want a middle ground: duplicates are forbidden unless the class body explicitly says that an override is intentional.

We will add a helper:

```python
allow_override("mode")
mode = "advanced"
```

Only the next overwrite of that named attribute will be accepted.

In [18]:
class ControlledOverrideNamespace(dict):
    def __init__(self):
        super().__init__()
        self.allowed_overrides = set()

    def allow_override(self, name):
        self.allowed_overrides.add(name)

    def __setitem__(self, key, value):
        if key in self:
            if key not in self.allowed_overrides:
                raise TypeError(
                    f"duplicate name {key!r}; call allow_override first"
                )

            self.allowed_overrides.remove(key)

        super().__setitem__(key, value)

As before, the metaclass inserts the temporary helper and removes it before creating the final class.

In [19]:
class ControlledOverrideMeta(type):
    @classmethod
    def __prepare__(mcls, name, bases, **kwargs):
        ns = ControlledOverrideNamespace()
        dict.__setitem__(ns, "allow_override", ns.allow_override)
        return ns

    def __new__(mcls, name, bases, namespace, **kwargs):
        normalized = dict(namespace)
        normalized.pop("allow_override", None)
        return super().__new__(mcls, name, bases, normalized)

Now let us make one overwrite explicit.

In [20]:
class RuntimeOptions(metaclass=ControlledOverrideMeta):
    mode = "safe"
    timeout = 10

    allow_override("mode")
    mode = "advanced"

check_equal(RuntimeOptions.mode, "advanced")
check_equal(RuntimeOptions.timeout, 10)
print(RuntimeOptions.mode)

advanced


An accidental duplicate still fails.

In [21]:
accidental_duplicate = '''
class BadRuntimeOptions(metaclass=ControlledOverrideMeta):
    timeout = 10
    timeout = 20
'''

try:
    exec(accidental_duplicate, globals(), globals())
except TypeError as exc:
    print("Caught:", exc)
else:
    raise AssertionError("expected TypeError")

Caught: duplicate name 'timeout'; call allow_override first


The key lesson is that the prepared namespace can preserve *intent* that would disappear from the final dictionary.

After class creation, both an accidental overwrite and an explicit overwrite look exactly the same: only the last value remains.

## Problem 7 — Intercept annotation-only declarations

There is a subtle case that ordinary `__setitem__` logging does not fully expose.

Consider:

```python
class Record:
    identifier: int
```

There is no ordinary value assignment to `identifier`.

Instead, Python records the annotation inside `__annotations__`.

Since `__prepare__` can seed the namespace, we can provide a custom object as `__annotations__` and observe writes into that nested mapping.

In [22]:
class AnnotationLog(dict):
    def __init__(self):
        super().__init__()
        self.events = []

    def __setitem__(self, key, value):
        self.events.append((key, value))
        super().__setitem__(key, value)

class AnnotationMeta(type):
    @classmethod
    def __prepare__(mcls, name, bases, **kwargs):
        return {"__annotations__": AnnotationLog()}

    def __new__(mcls, name, bases, namespace, **kwargs):
        annotation_log = namespace["__annotations__"]

        cls = super().__new__(mcls, name, bases, namespace)
        cls.__annotation_events__ = tuple(annotation_log.events)
        return cls

Now annotation-only declarations become individually visible.

In [23]:
class MessageRecord(metaclass=AnnotationMeta):
    identifier: int
    payload: bytes
    retry_count: int = 0

print("annotations:", MessageRecord.__annotations__)
print("events:", MessageRecord.__annotation_events__)

check_equal(
    [name for name, _ in MessageRecord.__annotation_events__],
    ["identifier", "payload", "retry_count"],
)

annotations: {'identifier': <class 'int'>, 'payload': <class 'bytes'>, 'retry_count': <class 'int'>}
events: (('identifier', <class 'int'>), ('payload', <class 'bytes'>), ('retry_count', <class 'int'>))


This is a deeper example of why the prepared namespace can matter.

Sometimes the information we want is stored *inside* one of the namespace values rather than through a direct assignment to the outer dictionary.

## Problem 8 — Return a custom mapping that is not a `dict`

The object returned by `__prepare__` needs to support the operations Python performs while executing the class body.

It does not have to be a literal `dict`.

However, there is another constraint later in the process: `type.__new__` expects a real dictionary-compatible namespace.

We will implement a small mapping from `MutableMapping` instead of inheriting from `dict`.

In [24]:
from collections.abc import MutableMapping

class ListBackedNamespace(MutableMapping):
    def __init__(self):
        self._data = {}
        self.write_order = []

    def __getitem__(self, key):
        return self._data[key]

    def __setitem__(self, key, value):
        if key not in self._data:
            self.write_order.append(key)
        self._data[key] = value

    def __delitem__(self, key):
        del self._data[key]

    def __iter__(self):
        return iter(self._data)

    def __len__(self):
        return len(self._data)

Python can execute a class body using this mapping.

But before calling `type.__new__`, we will convert it to an ordinary dictionary.

In [25]:
class ListBackedMeta(type):
    @classmethod
    def __prepare__(mcls, name, bases, **kwargs):
        return ListBackedNamespace()

    def __new__(mcls, name, bases, namespace, **kwargs):
        order = tuple(namespace.write_order)
        normalized = dict(namespace)

        cls = super().__new__(mcls, name, bases, normalized)
        cls.__write_order__ = order
        return cls

class ListBackedExample(metaclass=ListBackedMeta):
    alpha = 1
    beta = 2
    gamma = 3

print(ListBackedExample.__write_order__)
check("alpha" in ListBackedExample.__write_order__)

('__module__', '__qualname__', '__firstlineno__', 'alpha', 'beta', 'gamma', '__static_attributes__')


This separation is useful to remember:

- `__prepare__` controls the execution namespace,
- `type.__new__` controls class creation and has its own expectations.

A custom metaclass is responsible for bridging those two stages correctly.

## Problem 9 — Clean temporary names without breaking `super()`

Metaclasses sometimes normalize or filter a namespace before creating the class.

That is dangerous if the transformation accidentally removes `__classcell__`.

Python may need `__classcell__` when a method uses zero-argument `super()`.

We will create a metaclass that removes names beginning with `_temporary_`.

The safe approach is to start from the complete namespace and remove only the names we explicitly own.

In [26]:
class CleaningMeta(type):
    @classmethod
    def __prepare__(mcls, name, bases, **kwargs):
        return {}

    def __new__(mcls, name, bases, namespace, **kwargs):
        normalized = dict(namespace)

        temporary_names = [
            key
            for key in normalized
            if key.startswith("_temporary_")
        ]

        for key in temporary_names:
            del normalized[key]

        return super().__new__(mcls, name, bases, normalized)

Now we deliberately use zero-argument `super()`.

If we had rebuilt the namespace carelessly and lost `__classcell__`, class creation could fail.

In [27]:
class CleaningBase:
    def describe(self):
        return "base"

class CleaningChild(CleaningBase, metaclass=CleaningMeta):
    _temporary_note = "remove me"

    def describe(self):
        return super().describe() + " -> child"

check("_temporary_note" not in CleaningChild.__dict__)
check_equal(CleaningChild().describe(), "base -> child")
print(CleaningChild().describe())

base -> child


A useful rule is:

> When transforming a class namespace, remove only what you know you own. Do not casually rebuild the namespace from a whitelist of names you happen to recognize.

## Problem 10 — Combine `__prepare__` with descriptors

Descriptors already have a class-creation hook: `__set_name__`.

This gives us two useful moments:

- `__prepare__` can expose a convenient declaration helper,
- `__set_name__` can learn the final owner class and attribute name.

We will build a descriptor factory called `tracked(...)`.

The helper will exist only during class-body execution.

In [28]:
_MISSING = object()

class TrackedValue:
    def __init__(self, default=_MISSING):
        self.default = default
        self.name = None
        self.storage_name = None

    def __set_name__(self, owner, name):
        self.name = name
        self.storage_name = f"_{name}"

    def __get__(self, instance, owner):
        if instance is None:
            return self

        if hasattr(instance, self.storage_name):
            return getattr(instance, self.storage_name)

        if self.default is not _MISSING:
            return self.default

        raise AttributeError(self.name)

    def __set__(self, instance, value):
        setattr(instance, self.storage_name, value)

def tracked(default=_MISSING):
    return TrackedValue(default=default)

The metaclass injects `tracked` into the namespace and later removes the helper.

The descriptors themselves remain as real class attributes.

In [29]:
class TrackedMeta(type):
    @classmethod
    def __prepare__(mcls, name, bases, **kwargs):
        return {"tracked": tracked}

    def __new__(mcls, name, bases, namespace, **kwargs):
        normalized = dict(namespace)
        normalized.pop("tracked", None)

        tracked_names = tuple(
            key
            for key, value in normalized.items()
            if isinstance(value, TrackedValue)
        )

        cls = super().__new__(mcls, name, bases, normalized)
        cls.__tracked_names__ = tracked_names
        return cls

class Session(metaclass=TrackedMeta):
    user = tracked()
    state = tracked(default="new")

session = Session()
session.user = "alice"

check_equal(session.user, "alice")
check_equal(session.state, "new")
check_equal(Session.__tracked_names__, ("user", "state"))
check_equal(Session.user.name, "user")

print(Session.__tracked_names__)

('user', 'state')


Notice the division of responsibility.

`__prepare__` improves the declaration syntax.

The descriptor's `__set_name__` learns information that only exists once the actual class object is being created.

## Problem 11 — Cooperative `__prepare__` through metaclass inheritance

Advanced frameworks sometimes compose more than one metaclass feature.

For this to work, each metaclass should cooperate through `super()`.

We will create two metaclasses.

One adds a `feature_name`.

The other adds a `feature_version`.

Neither one should assume it is the only metaclass in the hierarchy.

In [30]:
class NameFeatureMeta(type):
    @classmethod
    def __prepare__(mcls, name, bases, **kwargs):
        namespace = super().__prepare__(name, bases, **kwargs)
        namespace["feature_name"] = name.lower()
        return namespace

class VersionFeatureMeta(type):
    @classmethod
    def __prepare__(mcls, name, bases, **kwargs):
        namespace = super().__prepare__(name, bases, **kwargs)
        namespace["feature_version"] = 1
        return namespace

Now we combine the two metaclasses.

The method-resolution order determines how the cooperative `super()` chain proceeds.

In [31]:
class CombinedFeatureMeta(NameFeatureMeta, VersionFeatureMeta):
    pass

class FeatureClass(metaclass=CombinedFeatureMeta):
    enabled = True

check_equal(FeatureClass.feature_name, "featureclass")
check_equal(FeatureClass.feature_version, 1)

print("metaclass MRO:")
for item in CombinedFeatureMeta.__mro__:
    print(" ", item)

metaclass MRO:
  <class '__main__.CombinedFeatureMeta'>
  <class '__main__.NameFeatureMeta'>
  <class '__main__.VersionFeatureMeta'>
  <class 'type'>
  <class 'object'>


The important point is not merely that multiple inheritance works.

The important point is that each metaclass layer asked its superclass for the namespace and then added only its own feature.

That makes the implementation composable.

## Problem 12 — Use `types.prepare_class` directly

The standard library exposes class-preparation machinery through `types.prepare_class`.

This lets us separate the three operations that a class statement normally performs for us:

1. select a metaclass,
2. prepare a namespace,
3. execute a body and create the class.

In [32]:
import types

class RuntimeMeta(type):
    @classmethod
    def __prepare__(mcls, name, bases, **kwargs):
        return {"origin": "prepared dynamically"}

    def __new__(mcls, name, bases, namespace, **kwargs):
        category = kwargs.pop("category", "general")
        cls = super().__new__(mcls, name, bases, namespace)
        cls.category = category
        return cls

First we ask Python to prepare everything needed for a class called `RuntimeGenerated`.

In [33]:
meta, runtime_namespace, remaining_keywords = types.prepare_class(
    "RuntimeGenerated",
    (),
    {
        "metaclass": RuntimeMeta,
        "category": "generated",
    },
)

print("resolved metaclass:", meta)
print("initial namespace:", runtime_namespace)
print("remaining keywords:", remaining_keywords)

resolved metaclass: <class '__main__.RuntimeMeta'>
initial namespace: {'origin': 'prepared dynamically'}
remaining keywords: {'category': 'generated'}


Now we execute a small class body inside the prepared namespace.

This is similar to what the `class` statement normally arranges for us.

In [34]:
runtime_body = '''
value = 21

def double(self):
    return self.value * 2
'''

exec(runtime_body, globals(), runtime_namespace)

RuntimeGenerated = meta(
    "RuntimeGenerated",
    (),
    runtime_namespace,
    **remaining_keywords,
)

check_equal(RuntimeGenerated.origin, "prepared dynamically")
check_equal(RuntimeGenerated.category, "generated")
check_equal(RuntimeGenerated().double(), 42)

print(RuntimeGenerated().double())

42


This is useful when a framework needs to construct classes from runtime specifications while still honoring normal metaclass preparation rules.

## Problem 13 — Build a decorator helper that exists only inside the class body

We will now combine several ideas.

The prepared namespace will expose a temporary decorator named `command`.

A method may be declared as:

```python
@command("start")
def start_job(self):
    ...
```

The helper marks the function.

The metaclass will later collect those marked functions into a command registry.

First we need the marker decorator.

It does not need to know the final class yet.

It only needs to attach metadata to the function object.

In [35]:
def command_marker(command_name):
    def decorate(func):
        func.__command_name__ = command_name
        return func
    return decorate

Now we can write a metaclass that injects the helper and removes it before the final class is created.

In [36]:
class CommandMeta(type):
    @classmethod
    def __prepare__(mcls, name, bases, **kwargs):
        return {"command": command_marker}

    def __new__(mcls, name, bases, namespace, **kwargs):
        normalized = dict(namespace)
        normalized.pop("command", None)

        inherited_commands = {}

        for base in bases:
            inherited_commands.update(
                getattr(base, "__commands__", {})
            )

        own_commands = {}

        for attribute_name, value in normalized.items():
            command_name = getattr(
                value,
                "__command_name__",
                None,
            )

            if command_name is None:
                continue

            if command_name in own_commands:
                raise TypeError(
                    f"duplicate command name: {command_name!r}"
                )

            own_commands[command_name] = attribute_name

        inherited_commands.update(own_commands)

        cls = super().__new__(
            mcls,
            name,
            bases,
            normalized,
        )
        cls.__commands__ = inherited_commands
        return cls

We will add a small base class that knows how to dispatch a command name to the registered method.

In [37]:
class CommandHandler(metaclass=CommandMeta):
    def dispatch(self, command_name, *args, **kwargs):
        try:
            method_name = self.__commands__[command_name]
        except KeyError:
            raise KeyError(
                f"unknown command: {command_name!r}"
            ) from None

        method = getattr(self, method_name)
        return method(*args, **kwargs)

Now the class declaration itself becomes very readable.

The `command` name is available while the body is executing, but it is not retained as a class attribute.

In [38]:
class JobHandler(CommandHandler):
    @command("start")
    def start_job(self, job_id):
        return f"starting {job_id}"

    @command("stop")
    def stop_job(self, job_id):
        return f"stopping {job_id}"

handler = JobHandler()

check_equal(
    handler.dispatch("start", 17),
    "starting 17",
)
check_equal(
    handler.dispatch("stop", 17),
    "stopping 17",
)
check("command" not in JobHandler.__dict__)

print(JobHandler.__commands__)

{'start': 'start_job', 'stop': 'stop_job'}


The registry also supports inheritance because `CommandMeta.__new__` starts with registrations from the base classes.

In [39]:
class ExtendedJobHandler(JobHandler):
    @command("status")
    def status(self, job_id):
        return f"status for {job_id}"

extended = ExtendedJobHandler()

check_equal(
    extended.dispatch("start", 5),
    "starting 5",
)
check_equal(
    extended.dispatch("status", 5),
    "status for 5",
)

print(ExtendedJobHandler.__commands__)

{'start': 'start_job', 'stop': 'stop_job', 'status': 'status'}


This is a realistic use of `__prepare__`.

The metaclass is not merely injecting arbitrary attributes.

It is shaping the *language used inside the class body* and then translating that declaration syntax into runtime metadata.

## Problem 14 — Capstone: a staged workflow DSL

For the final problem, we will build a small workflow framework.

A workflow class should read naturally:

```python
class ImportWorkflow(Workflow):
    stage("extract")

    @step()
    def read_source(self):
        ...

    stage("transform")

    @step()
    def normalize(self):
        ...
```

We want the framework to remember both:

- which stage each step belongs to,
- the declaration order of the steps.

This problem is more subtle than the command registry.

The temporary `stage(...)` helper changes namespace state.

The temporary `step()` decorator needs to capture the current stage at the moment the decorator expression is evaluated.

We will let the namespace provide both helpers.

The namespace knows the current stage, so it can manufacture a decorator carrying that stage information.

In [40]:
class WorkflowNamespace(dict):
    def __init__(self):
        super().__init__()
        self.current_stage = None
        self.step_declaration_order = []

    def select_stage(self, stage_name):
        self.current_stage = stage_name

    def make_step_decorator(self):
        stage_at_declaration = self.current_stage

        if stage_at_declaration is None:
            raise RuntimeError(
                "call stage(...) before declaring a workflow step"
            )

        def decorate(func):
            func.__workflow_stage__ = stage_at_declaration
            return func

        return decorate

    def __setitem__(self, key, value):
        if getattr(value, "__workflow_stage__", None) is not None:
            self.step_declaration_order.append(key)

        super().__setitem__(key, value)

The metaclass will inject `stage` and `step`.

After the class body has finished, it will remove those helpers and construct an ordered workflow plan.

In [41]:
class WorkflowMeta(type):
    @classmethod
    def __prepare__(mcls, name, bases, **kwargs):
        ns = WorkflowNamespace()

        dict.__setitem__(ns, "stage", ns.select_stage)
        dict.__setitem__(ns, "step", ns.make_step_decorator)

        return ns

    def __new__(mcls, name, bases, namespace, **kwargs):
        declaration_order = tuple(
            namespace.step_declaration_order
        )

        normalized = dict(namespace)
        normalized.pop("stage", None)
        normalized.pop("step", None)

        inherited_plan = []

        for base in bases:
            inherited_plan.extend(
                getattr(base, "__workflow_plan__", ())
            )

        own_plan = []

        for method_name in declaration_order:
            func = normalized[method_name]
            own_plan.append(
                (
                    func.__workflow_stage__,
                    method_name,
                )
            )

        cls = super().__new__(
            mcls,
            name,
            bases,
            normalized,
        )

        cls.__workflow_plan__ = tuple(
            inherited_plan + own_plan
        )

        return cls

Now we need a base class that can execute the plan.

For the tutorial we will keep the execution model deliberately simple: call each step in declaration order and collect the results.

In [42]:
class Workflow(metaclass=WorkflowMeta):
    def run(self):
        results = []

        for stage_name, method_name in self.__workflow_plan__:
            method = getattr(self, method_name)
            results.append(
                (stage_name, method_name, method())
            )

        return results

Let us define a concrete workflow.

Notice how little framework machinery appears in the final class body.

In [43]:
class ImportWorkflow(Workflow):
    stage("extract")

    @step()
    def read_source(self):
        return "source-data"

    @step()
    def parse_rows(self):
        return "parsed"

    stage("transform")

    @step()
    def normalize(self):
        return "normalized"

    stage("load")

    @step()
    def write_database(self):
        return "written"

workflow = ImportWorkflow()

print(ImportWorkflow.__workflow_plan__)
print(workflow.run())

check_equal(
    ImportWorkflow.__workflow_plan__,
    (
        ("extract", "read_source"),
        ("extract", "parse_rows"),
        ("transform", "normalize"),
        ("load", "write_database"),
    ),
)

(('extract', 'read_source'), ('extract', 'parse_rows'), ('transform', 'normalize'), ('load', 'write_database'))
[('extract', 'read_source', 'source-data'), ('extract', 'parse_rows', 'parsed'), ('transform', 'normalize', 'normalized'), ('load', 'write_database', 'written')]


The workflow now contains information that would have been awkward to reconstruct from the final class dictionary alone.

The class body expressed transitions between stages.

The prepared namespace captured those transitions while the body was running.

Let us also verify the framework fails early when a step is declared before any stage has been selected.

In [44]:
bad_workflow_source = '''
class BadWorkflow(Workflow):
    @step()
    def orphan_step(self):
        return None
'''

try:
    exec(bad_workflow_source, globals(), globals())
except RuntimeError as exc:
    print("Caught:", exc)
else:
    raise AssertionError("expected RuntimeError")

Caught: call stage(...) before declaring a workflow step


This is a good capstone example because several concepts are working together:

- `__prepare__` creates a stateful namespace,
- the namespace exposes temporary helpers,
- one helper changes namespace state,
- another helper captures that state,
- `__setitem__` records declaration order,
- metaclass `__new__` translates the temporary declaration model into permanent class metadata.

## A few more experiments

The following short exercises are intentionally left as extensions.

They are useful because each one changes only one part of the machinery we have already built.

### Extension A — Stage validation

Modify `WorkflowNamespace.select_stage` so that the same stage cannot be entered twice.

Then decide whether that rule belongs in the namespace or in `WorkflowMeta.__new__`.

There is no universal answer: choose based on whether the rule concerns the *sequence of body execution* or only the final collected metadata.

### Extension B — Command aliases

Extend `command_marker` so one method may register multiple command names.

For example:

```python
@command("remove", "delete", "rm")
def remove_job(self, job_id):
    ...
```

Think carefully about when duplicate alias names should be rejected.

### Extension C — Annotation policy

Extend `AnnotationLog` so that private annotations are allowed but ignored by the final field registry.

Then compare two designs:

- filtering while annotations are written,
- collecting everything and filtering in metaclass `__new__`.

### Extension D — Method instrumentation policy

Our instrumenting namespace wrapped every ordinary function.

Change it so that only methods whose names start with `tracked_` are wrapped.

This is less magical and often easier for users to understand.

### Extension E — Cooperative class keywords

Add a class-header keyword to one of the cooperative metaclasses.

For example:

```python
class Feature(
    metaclass=CombinedFeatureMeta,
    version=3,
):
    ...
```

Consume only the keyword owned by the relevant metaclass layer and keep `super()` cooperation intact.

## Summary

The central idea of `__prepare__` is not simply that it returns a dictionary.

The important idea is **timing**.

The returned object participates while Python executes the class body.

That makes `__prepare__` especially useful when a framework needs information that would otherwise disappear before `__new__` receives the completed namespace.

The examples in this notebook showed several categories of problems:

- early validation,
- assignment interception,
- controlled overwrites,
- nested annotation tracking,
- temporary DSL helpers,
- stateful declaration syntax,
- descriptor integration,
- safe namespace transformation,
- cooperative metaclasses,
- dynamic class creation.

And there is an equally important design lesson:

Use `__prepare__` only when the problem really depends on class-body execution.

If the same job can be handled more simply after the class exists, a descriptor, class decorator, or `__init_subclass__` may be the better tool.